In [1]:
# clear namespace so notebook starts fresh (no leftover variables/etc. from previous notebooks/executions)
%reset -f
# import arcpy library
import arcpy
# clear any leftover geoprocessing env settings from previous notebooks/executions
arcpy.ResetEnvironments()

In [2]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Input study area boundary polygon to calculate geometry attributes for (* adds columns to input feature class Attribute Table, no new output feature classes created)
boundary = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb\boundary_thesis_p26920"

# Output CSV of boundary feature class Attribute Table with added fields for extent/area
boundary_csv = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\boundary_thesis_p26920_atttable.csv"

In [3]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [4]:
# ── 1. ADD COLUMNS TO ATTRIBUTE TABLE WITH MIN/MAX LON AND LAT ───────────────────────────────────────

# Calculate Geometry Attributes (Data Management Tools):
try:
    arcpy.management.CalculateGeometryAttributes(
        in_features=boundary,
        geometry_property=[
        #   ["desired field name",  "calculated property"]
            ["x_coordinate_min_dd", "EXTENT_MIN_X"],
            ["x_coordinate_max_dd", "EXTENT_MAX_X"],
            ["y_coordinate_min_dd", "EXTENT_MIN_Y"],
            ["y_coordinate_max_dd", "EXTENT_MAX_Y"],
            ["area_geodesic_m2",     "AREA_GEODESIC"]
        ],
        # output geodesic area of polygon in square meters
        area_unit="SQUARE_METERS",
        # defaults to CS of input feature class (** projected EPSG:26920 will output min/max x and y coordinates in meters Easting/Northing which are not very easily interpretable so set CS for coordinates/area calculations to input polygon's *geographic* CS (EPSG:4269) to output coordinate values in decimal degrees without changing datums = *NO* reprojection needed)
        coordinate_system=arcpy.SpatialReference(4269)
        # coordinate_format leave as default ("SAME_AS_INPUT") since set coordinate_system to geographic CS output values will default to DD
    )
    print(f"✅ Calculate Geometry Attributes tool ran successfully \n")

# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] Calculate Geometry Attributes tool failed: {e} \n")

✅ Calculate Geometry Attributes tool ran successfully 



In [5]:
# ── INSPECT NEW FIELDS IN ATTRIBUTE TABLE ────────────────────────────────────────────────────────────

# Read data from Attribute Table for fields created by Calculate Geometry Attributes tool to confirm all values populated successfully
fields = ["x_coordinate_min_dd", "x_coordinate_max_dd", "y_coordinate_min_dd", "y_coordinate_max_dd", "area_geodesic_m2"]

# Grab values in the single row of feature class Attribute Table
with arcpy.da.SearchCursor(boundary, fields) as cursor:
    row = next(cursor)

# Print message with field names and values, or warning message if any field returned null
if all(v is not None for v in row):
    print(f"✅ All fields successfully populated: \n")
    for name, val in zip(fields, row):
        print(f"  '{name}': {val}")
else:
    print(f"⚠️[WARNING] One or more fields are null: \n")
    for name, val in zip(fields, row):
        print(f"  '{name}': {val}")

✅ All fields successfully populated: 

  'x_coordinate_min_dd': -68.5223638392393
  'x_coordinate_max_dd': -61.098875148075884
  'y_coordinate_min_dd': 15.557943008270607
  'y_coordinate_max_dd': 20.41715308164909
  'area_geodesic_m2': 411324336691.72003


In [6]:
# ── 2. EXPORT ATTRIBUTE TABLE TO CSV ─────────────────────────────────────────────────────────────────

# Export Table (Conversion Tools):
try:
    arcpy.conversion.ExportTable(
        in_table=boundary,
        out_table=boundary_csv
    )
    print(f"✅ Attribute Table data exported to CSV: {boundary_csv} \n")

# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] CSV export failed: {e} \n")

✅ Attribute Table data exported to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\boundary_thesis_p26920_atttable.csv 

